# ACCESSAI: First Proto

In [1]:
from ultralytics import YOLO

import torch
import os
import yaml
import requests
import random
import shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
import albumentations as A

from pathlib import Path
from collections import Counter

/home/JNB/anaconda3/envs/yoloEnv/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
print(f"¿GPU disponible? {torch.cuda.is_available()}")
print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
print(f"Versión HIP: {torch.version.hip}")

¿GPU disponible? True
Dispositivo: AMD Radeon RX 9060 XT
Versión HIP: 7.13.99004


## ROD Dataset + Curbs Dataset: detección de objetos

In [3]:
ROD_PATH = "datasets/rod_25cls/" 
CURBS_PATH = "datasets/curbs/"

### Comprobar estructura de los datasets

In [4]:
def show_dataset(dataset_path, name):

    print(f"ESTRUCTURA DEL DATASET: {name}")

    dataset = Path(dataset_path)

    if not dataset.exists():
        print(f"ERROR: la ruta no existe")
        return

    for split in ["train", "valid", "test"]:
        split_dir = dataset / split

        if not split_dir.exists():
            print(f"  {split:5s}: NO EXISTE")
            continue

        img_dir = split_dir / "images"
        lbl_dir = split_dir / "labels"

        n_imgs = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
        n_lbls = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0

        print(f"  {split:5s}: {n_imgs:>6d} imágenes, {n_lbls:>6d} labels")

In [5]:
show_dataset(ROD_PATH, "ROD")

ESTRUCTURA DEL DATASET: ROD
  train:  19186 imágenes,  19186 labels
  valid:   3511 imágenes,   3511 labels
  test :   1629 imágenes,   1629 labels


In [6]:
show_dataset(CURBS_PATH, "Curbs")

ESTRUCTURA DEL DATASET: Curbs
  train:    955 imágenes,    955 labels
  valid:    270 imágenes,    270 labels
  test :    207 imágenes,    207 labels


### Distribución de instancias por clase

In [7]:
def show_class_distribution(dataset_path, name):
    # Distribución de clases en el split de train
    print(f"DISTRIBUCIÓN DE CLASES — {name} (train)")

    # Leer el data.yaml para obtener los nombres de clase
    yaml_path = os.path.join(dataset_path, "data.yaml")
    
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    names = data_config.get('names', [])

    # Contar instancias en train
    train_lbl_dir = os.path.join(dataset_path, "train", "labels")
    if not os.path.exists(train_lbl_dir):
        print(f"  No existe el directorio: {train_lbl_dir}")
        return

    contador = Counter()
    for lbl_file in os.listdir(train_lbl_dir):
        with open(os.path.join(train_lbl_dir, lbl_file)) as f:
            for line in f:
                if line.strip():
                    contador[int(line.split()[0])] += 1

    total = sum(contador.values())

    # Imprimir tabla ordenada por número de instancias (descendente)
    print(f"{'#':>3s}  {'Clase':25s} {'Instancias':>10s} {'%':>7s}")
    print("-" * 50)
    
    for i, count in sorted(contador.items(), key=lambda x: -x[1]):
        nombre = names[i] if i < len(names) else f"Clase {i}"
        pct = count / total * 100 if total > 0 else 0
        print(f"{i:>3d}  {nombre:25s} {count:>10d} {pct:>6.2f}%")
        
    print("-" * 50)
    print(f"{'':>3s}  {'TOTAL':25s} {total:>10d}")


In [8]:
show_class_distribution(ROD_PATH, "ROD-25cls")
show_class_distribution(CURBS_PATH, "Curbs")

DISTRIBUCIÓN DE CLASES — ROD-25cls (train)
  #  Clase                     Instancias       %
--------------------------------------------------
  3  Person                          7609  23.59%
  2  Car                             4328  13.42%
 14  Pedestrian crosswalk            1875   5.81%
 11  Manhole                         1779   5.52%
 23  Chair                           1757   5.45%
 12  Tree                            1632   5.06%
 13  Guard rail                      1385   4.29%
  9  Dustbin                         1351   4.19%
  4  Stairs                          1125   3.49%
 16  Bus                             1009   3.13%
  8  Motorcycle                       888   2.75%
  1  Building                         886   2.75%
 17  Bench                            706   2.19%
  5  Traffic sign                     689   2.14%
 18  Traffic Cone                     678   2.10%
 10  Dog                              633   1.96%
 15  Truck                            621   1.93%
  7  R

#### Ratio original ROD-DATASET con 25 clases - 44:1

### Conversión de Curbs: de segmentación a detección con bbs

#### (Solo es necesario convertir el dataset de Curbs una vez. Una vez esté hecho no hace falta volver a ejecutar esta celda)

In [ ]:
# EJECUTAR SI EL DATASET DE CURBS NO ESTÁ CONVERTIDO

INPUT = Path("datasets/curbs")
OUTPUT = Path("datasets/curbs_yolo_det")


CLASE_PROVISIONAL = 0

def poligono_a_bbox(coords):
    xs = coords[0::2]  # coordenadas x
    ys = coords[1::2]  # coordenadas y

    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    cx = (x_min + x_max) / 2
    cy = (y_min + y_max) / 2
    w = x_max - x_min
    h = y_max - y_min

    return cx, cy, w, h


def procesar_split(split_name):
    src_img = INPUT / split_name / "images"
    src_lbl = INPUT / split_name / "labels"
    dst_img = OUTPUT / split_name / "images"
    dst_lbl = OUTPUT / split_name / "labels"

    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    n_imgs = 0
    n_labels_converted = 0
    n_instancias_validas = 0
    n_instancias_invalidas = 0

    for lbl_path in src_lbl.glob("*.txt"):
        nuevas_lineas = []

        with open(lbl_path) as f:
            for line in f:
                partes = line.strip().split()
                if len(partes) < 5:
                    continue  # línea vacía o mal formada

                # Detectar tipo: 5 valores = detección, ≥7 = segmentación
                n_valores = len(partes)

                if n_valores == 5:
                    # Ya es bbox, solo renombrar clase
                    nuevas_lineas.append(
                        f"{CLASE_PROVISIONAL} {' '.join(partes[1:])}"
                    )
                    n_instancias_validas += 1
                elif n_valores >= 7 and n_valores % 2 == 1:
                    # Es polígono (1 + 2*N valores)
                    coords = [float(x) for x in partes[1:]]
                    cx, cy, w, h = poligono_a_bbox(coords)
                    nuevas_lineas.append(
                        f"{CLASE_PROVISIONAL} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
                    )
                    n_instancias_validas += 1
                else:
                    n_instancias_invalidas += 1

        if not nuevas_lineas:
            continue  # no escribir label vacío

        # Copiar imagen
        for ext in [".jpg", ".jpeg", ".png"]:
            img_path = src_img / (lbl_path.stem + ext)
            if img_path.exists():
                shutil.copy2(img_path, dst_img / img_path.name)
                break

        # Escribir label convertido
        with open(dst_lbl / lbl_path.name, "w") as f:
            f.write("\n".join(nuevas_lineas))

        n_imgs += 1
        n_labels_converted += 1

    print(f"{split_name:6s} → {n_imgs} imgs procesadas, "
          f"{n_labels_converted} labels escritos, "
          f"{n_instancias_validas} instancias válidas, "
          f"{n_instancias_invalidas} inválidas")


print("Conversión curbs (segmentación) → curbs_yolo_det (detección) \n")

for split in ["train", "valid", "test"]:
    procesar_split(split)

print(f"\n✓ Dataset convertido en: {OUTPUT}/")

### Mapeo de ROD a 13 clases: eliminando clases irrelevantes

In [9]:
ROD_PATH = Path("datasets/rod_25cls")
OUTPUT = Path("datasets/rod_14cls")


# Presernamos las 13 clases más importantes del dataset original

MAPEO_ROD = {
    0: 0,    # Bike → Bike
    1: -1,   # Building → ELIMINAR
    2: 1,    # Car → Car
    3: 2,    # Person → Person
    4: 3,    # Stairs → Stairs
    5: 4,    # Traffic sign → Traffic sign
    6: -1,   # Electrical Pole → ELIMINAR
    7: -1,   # Road → ELIMINAR
    8: 5,    # Motorcycle → Motorcycle
    9: 6,    # Dustbin → Dustbin
    10: -1,  # Dog → ELIMINAR
    11: 7,   # Manhole → Manhole
    12: -1,  # Tree → ELIMINAR
    13: -1,  # Guard rail → ELIMINAR
    14: 8,   # Pedestrian crosswalk → Pedestrian crosswalk
    15: 9,   # Truck → Truck
    16: 10,  # Bus → Bus
    17: 11,  # Bench → Bench
    18: -1,  # Traffic Cone → ELIMINAR
    19: -1,  # Fire hydrant → ELIMINAR
    20: -1,  # Traffic Barrel → ELIMINAR
    21: 12,  # Plant Pot → Plant Pot
    22: -1,  # Electrical Box → ELIMINAR
    23: -1,  # Chair → ELIMINAR
    24: -1,  # Bicycle Rack → ELIMINAR
}

In [10]:
def aplicar_mapeo_rod(src_root, dst_root):
    
    #Aplica el mapeo 25 → 13 a ROD-25cls.
    #Convierte líneas de segmentación a detección (bbox) para no perder instancias.
    
    for split in ["train", "valid", "test"]:
        src_img = src_root / split / "images"
        src_lbl = src_root / split / "labels"
        dst_img = dst_root / split / "images"
        dst_lbl = dst_root / split / "labels"

        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)

        n_imgs = 0
        n_inst_validas = 0
        n_inst_eliminadas = 0
        n_convertidas = 0
        n_invalidas = 0

        for lbl_path in src_lbl.glob("*.txt"):
            nuevas_lineas = []

            with open(lbl_path) as f:
                for line in f:
                    partes = line.strip().split()
                    if len(partes) < 5:
                        n_invalidas += 1
                        continue

                    clase_orig = int(partes[0])
                    nueva = MAPEO_ROD.get(clase_orig, -1)

                    if nueva == -1:
                        n_inst_eliminadas += 1
                        continue

                    if len(partes) == 5:
                        # Formato detección
                        nuevas_lineas.append(f"{nueva} {' '.join(partes[1:])}")
                        n_inst_validas += 1
                    else:
                        # Formato segmentación: convertir a bbox
                        coords = [float(x) for x in partes[1:]]
                        xs = coords[0::2]
                        ys = coords[1::2]

                        x_min, x_max = min(xs), max(xs)
                        y_min, y_max = min(ys), max(ys)

                        cx = (x_min + x_max) / 2
                        cy = (y_min + y_max) / 2
                        w = x_max - x_min
                        h = y_max - y_min

                        nuevas_lineas.append(f"{nueva} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                        n_inst_validas += 1
                        n_convertidas += 1

            if not nuevas_lineas:
                continue

            # Copiar imagen
            for ext in [".jpg", ".jpeg", ".png"]:
                img = src_img / (lbl_path.stem + ext)
                if img.exists():
                    shutil.copy2(img, dst_img / img.name)
                    break

            # Escribir label convertido
            with open(dst_lbl / lbl_path.name, "w") as f:
                f.write("\n".join(nuevas_lineas))

            n_imgs += 1

        print(f"  {split:6s} → {n_imgs} imgs, {n_inst_validas} inst. válidas, "
              f"{n_inst_eliminadas} eliminadas, {n_convertidas} segment→bbox, "
              f"{n_invalidas} inválidas")

In [11]:
aplicar_mapeo_rod(ROD_PATH, OUTPUT)  # Mapeo ROD

  train  → 13064 imgs, 22863 inst. válidas, 9388 eliminadas, 2139 segment→bbox, 0 inválidas
  valid  → 2443 imgs, 4114 inst. válidas, 1458 eliminadas, 552 segment→bbox, 0 inválidas
  test   → 1049 imgs, 1616 inst. válidas, 756 eliminadas, 285 segment→bbox, 0 inválidas


In [12]:
ROD_13CLS = Path("datasets/rod_14cls")  # dataset mapeado, ANTES de fusión con Curbs

# Distribución de clases en train
contador = Counter()
for lbl in (ROD_13CLS / "train" / "labels").glob("*.txt"):
    with open(lbl) as f:
        for line in f:
            if line.strip():
                contador[int(line.split()[0])] += 1

nombres_13 = ['Bike', 'Car', 'Person', 'Stairs', 'Traffic sign', 'Motorcycle',
              'Dustbin', 'Manhole', 'Pedestrian crosswalk', 'Truck', 'Bus',
              'Bench', 'Plant Pot']

total = sum(contador.values())
print(f"=== Distribución ROD-13cls (train, antes de fusión con Curbs) ===")
print(f"{'#':>3s}  {'Clase':25s} {'Instancias':>10s} {'%':>7s}")
print("-" * 50)
for i, nombre in enumerate(nombres_13):
    c = contador.get(i, 0)
    pct = c / total * 100 if total > 0 else 0
    print(f"{i:>3d}  {nombre:25s} {c:>10d} {pct:>6.2f}%")
print("-" * 50)
print(f"{'':>3s}  {'TOTAL':25s} {total:>10d}")

print(f"\nImágenes train: {len(list((ROD_13CLS / 'train' / 'images').glob('*')))}")
print(f"Labels train: {len(list((ROD_13CLS / 'train' / 'labels').glob('*.txt')))}")

=== Distribución ROD-13cls (train, antes de fusión) ===
  #  Clase                     Instancias       %
--------------------------------------------------
  0  Bike                             323   1.41%
  1  Car                             4328  18.93%
  2  Person                          7609  33.28%
  3  Stairs                          1125   4.92%
  4  Traffic sign                     689   3.01%
  5  Motorcycle                       888   3.88%
  6  Dustbin                         1351   5.91%
  7  Manhole                         1779   7.78%
  8  Pedestrian crosswalk            1875   8.20%
  9  Truck                            621   2.72%
 10  Bus                             1009   4.41%
 11  Bench                            706   3.09%
 12  Plant Pot                        560   2.45%
--------------------------------------------------
     TOTAL                          22863

Imágenes train: 13064
Labels train: 13064


### Fusión del dataset ROD reducido con Curbs: ROD_14cls

In [13]:
CURBS_PATH = Path("datasets/curbs_yolo_det")
CLASE_CURB = 13  

def fusionar_curbs(curbs_root, dst_root):
    # Añade las imágenes de Curbs al dataset base, con la clase CLASE_CURB
    
    for split in ["train", "valid", "test"]:
        src_img = curbs_root / split / "images"
        src_lbl = curbs_root / split / "labels"
        dst_img = dst_root / split / "images"
        dst_lbl = dst_root / split / "labels"

        n_imgs = 0
        n_inst = 0

        for lbl_path in src_lbl.glob("*.txt"):
            nuevas_lineas = []

            with open(lbl_path) as f:
                for line in f:
                    partes = line.strip().split()
                    if len(partes) != 5:
                        continue
                    nuevas_lineas.append(f"{CLASE_CURB} {' '.join(partes[1:])}")
                    n_inst += 1

            if not nuevas_lineas:
                continue

            # Copiar imagen con prefijo para evitar colisiones
            for ext in [".jpg", ".jpeg", ".png"]:
                img = src_img / (lbl_path.stem + ext)
                if img.exists():
                    nuevo_nombre = f"curb_{img.name}"
                    shutil.copy2(img, dst_img / nuevo_nombre)
                    break

            # Escribir label con el mismo prefijo
            with open(dst_lbl / f"curb_{lbl_path.name}", "w") as f:
                f.write("\n".join(nuevas_lineas))

            n_imgs += 1

        print(f"  {split:6s} → {n_imgs} imgs, {n_inst} instancias de Bordillo")

In [14]:
# Creación del data.yaml para el nuevo dataset de 14 clases

def crear_data_yaml(output_root):
    yaml_content = """train: ../train/images
val: ../valid/images
test: ../test/images

nc: 14
names: ['Bike', 'Car', 'Person', 'Stairs', 'Traffic sign', 'Motorcycle', 'Dustbin', 'Manhole', 'Pedestrian crosswalk', 'Truck', 'Bus', 'Bench', 'Plant Pot', 'Curb']
"""

    with open(OUTPUT / "data.yaml", "w") as f:
        f.write(yaml_content)

    print(f"✓ data.yaml creado en: {OUTPUT / 'data.yaml'}")
    print("\nContenido:")
    print(yaml_content)

In [15]:
fusionar_curbs(CURBS_PATH, OUTPUT) # Fusión con Curbs
crear_data_yaml(OUTPUT)  # Creación de data.yaml de dataset definitivo

  train  → 955 imgs, 1113 instancias de Bordillo
  valid  → 270 imgs, 316 instancias de Bordillo
  test   → 207 imgs, 237 instancias de Bordillo
✓ data.yaml creado en: datasets/rod_14cls/data.yaml

Contenido:
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 14
names: ['Bike', 'Car', 'Person', 'Stairs', 'Traffic sign', 'Motorcycle', 'Dustbin', 'Manhole', 'Pedestrian crosswalk', 'Truck', 'Bus', 'Bench', 'Plant Pot', 'Curb']



In [17]:
print("Estructura del nuevo dataset \n")
for split in ["train", "valid", "test"]:
    imgs = list((OUTPUT / split / "images").glob("*"))
    lbls = list((OUTPUT / split / "labels").glob("*.txt"))
    print(f"  {split:6s}: {len(imgs):>6d} imgs, {len(lbls):>6d} labels")

print("\n Distribución de clases (train) \n")
contador = Counter()
for lbl in (OUTPUT / "train" / "labels").glob("*.txt"):
    with open(lbl) as f:
        for line in f:
            if line.strip():
                contador[int(line.split()[0])] += 1

names = ['Bike', 'Car', 'Person', 'Stairs', 'Traffic sign', 'Motorcycle',
         'Dustbin', 'Manhole', 'Pedestrian crosswalk', 'Truck', 'Bus',
         'Bench', 'Plant Pot', 'Curb']

total = sum(contador.values())
for i, nombre in enumerate(names):
    c = contador.get(i, 0)
    print(f"  [{i:>2d}] {nombre:25s} {c:>7d}  ({c/total*100:5.2f}%)")
print(f"  {'':28s} {'TOTAL':>7s}: {total}")


mayor = max(contador.values())
menor = min(contador.values())

print(f"\nRatio mayor/menor: {mayor/menor:.1f}:1")

Estructura del nuevo dataset 

  train :  14019 imgs,  14019 labels
  valid :   2713 imgs,   2713 labels
  test  :   1256 imgs,   1256 labels

 Distribución de clases (train) 

  [ 0] Bike                          323  ( 1.35%)
  [ 1] Car                          4328  (18.05%)
  [ 2] Person                       7609  (31.74%)
  [ 3] Stairs                       1125  ( 4.69%)
  [ 4] Traffic sign                  689  ( 2.87%)
  [ 5] Motorcycle                    888  ( 3.70%)
  [ 6] Dustbin                      1351  ( 5.63%)
  [ 7] Manhole                      1779  ( 7.42%)
  [ 8] Pedestrian crosswalk         1875  ( 7.82%)
  [ 9] Truck                         621  ( 2.59%)
  [10] Bus                          1009  ( 4.21%)
  [11] Bench                         706  ( 2.94%)
  [12] Plant Pot                     560  ( 2.34%)
  [13] Curb                         1113  ( 4.64%)
                                 TOTAL: 23976

Ratio mayor/menor: 23.6:1


In [ ]:
DATASET_SRC = Path("datasets/rod_5cls")
DATASET_DST = Path("datasets/rod_5cls_augmented")

# [2] Barrera_Arquitectonica, [3] Bordillo, [4] Elemento_Peatonal
CLASES_MINORITARIAS = [2, 3, 4]
FACTOR_AUMENTO = 3 # Versiones extra por imagen original

random.seed(42)

# Transformaciones de augmentation
TRANSFORM = A.Compose([
    A.HorizontalFlip(p=0.5), # Flip hortizontal
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.4), # Brillo y contraste
    A.Rotate(limit=5, p=0.3, border_mode=cv2.BORDER_REFLECT_101),  # Rotación
], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.3,
))

In [ ]:
def leer_label(lbl_path):
    # Lee un archivo .txt y devuelve listas de clases y bboxes
    
    clases = []
    bboxes = []
    
    with open(lbl_path) as f:
        for line in f:
            partes = line.strip().split()
            if len(partes) != 5:
                continue
            clases.append(int(partes[0]))
            bboxes.append([float(x) for x in partes[1:]])
            
    return clases, bboxes

In [ ]:
def escribir_label(lbl_path, clases, bboxes):
    # Escribe un archivo .txt con clases y bboxes
    
    with open(lbl_path, "w") as f:
        for c, b in zip(clases, bboxes):
            f.write(f"{int(c)} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}\n")

In [ ]:
def copiar_split(split_name):
    # Copia un split completo (train/valid/test) al dataset destino
    
    src_img = DATASET_SRC / split_name / "images"
    src_lbl = DATASET_SRC / split_name / "labels"
    dst_img = DATASET_DST / split_name / "images"
    dst_lbl = DATASET_DST / split_name / "labels"

    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    n_copiadas = 0
    for lbl_path in src_lbl.glob("*.txt"):
        # Buscar la imagen asociada
        img_found = None
        for ext in [".jpg", ".jpeg", ".png"]:
            img_path = src_img / (lbl_path.stem + ext)
            if img_path.exists():
                img_found = img_path
                break
        if img_found is None:
            continue

        shutil.copy2(img_found, dst_img / img_found.name)
        shutil.copy2(lbl_path, dst_lbl / lbl_path.name)
        n_copiadas += 1

    print(f"  {split_name:6s}: {n_copiadas} imágenes copiadas")
    
    return n_copiadas

In [ ]:
def aumentar_train():
    # Genera imágenes aumentadas de las clases minoritarias en train
    
    src_img = DATASET_SRC / "train" / "images"
    src_lbl = DATASET_SRC / "train" / "labels"
    dst_img = DATASET_DST / "train" / "images"
    dst_lbl = DATASET_DST / "train" / "labels"

    n_originales_con_min = 0
    n_aug_generadas = 0
    n_fallidas = 0

    for lbl_path in src_lbl.glob("*.txt"):
        clases, bboxes = leer_label(lbl_path)

        # ¿Contiene alguna clase minoritaria?
        if not any(c in CLASES_MINORITARIAS for c in clases):
            continue

        n_originales_con_min += 1

        # Buscar imagen
        img_path = None
        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = src_img / (lbl_path.stem + ext)
            if candidate.exists():
                img_path = candidate
                break
        if img_path is None:
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        # Generar N versiones aumentadas
        for i in range(FACTOR_AUMENTO):
            try:
                transformed = TRANSFORM(image=img, bboxes=bboxes, class_labels=clases)
            except Exception as e:
                n_fallidas += 1
                continue

            aug_img = transformed["image"]
            aug_bboxes = transformed["bboxes"]
            aug_clases = transformed["class_labels"]

            if not aug_bboxes:
                n_fallidas += 1
                continue

            out_name = f"{lbl_path.stem}_aug{i}"
            cv2.imwrite(str(dst_img / f"{out_name}.jpg"), aug_img)
            escribir_label(dst_lbl / f"{out_name}.txt", aug_clases, aug_bboxes)
            n_aug_generadas += 1

    print(f"  Imágenes con clase minoritaria: {n_originales_con_min}")
    print(f"  Versiones aumentadas generadas: {n_aug_generadas}")
    print(f"  Fallidas (artefactos): {n_fallidas}")

In [ ]:
def crear_data_yaml():
    # Crea el data.yaml del dataset aumentado
    
    yaml_content = """train: ../train/images
val: ../valid/images
test: ../test/images

nc: 5
names: ['Obstaculo_Dinamico', 'Obstaculo_Fijo', 'Barrera_Arquitectonica', 'Bordillo', 'Elemento_Peatonal']
"""
    
    with open(DATASET_DST / "data.yaml", "w") as f:
        f.write(yaml_content)
        
    print(f"  data.yaml creado en: {DATASET_DST / 'data.yaml'}")

In [ ]:
print("AUGMENTATION SELECTIVO + FUSIÓN")

# Limpiar el dataset destino si existe
if DATASET_DST.exists():
    print(f"\n El dataset destino ya existe. Eliminando...")
    shutil.rmtree(DATASET_DST)
    print(f"  Eliminado: {DATASET_DST}")

print(f"\n1. Copiando dataset original a {DATASET_DST}...")
for split in ["train", "valid", "test"]:
    copiar_split(split)

print(f"\n2. Generando imágenes aumentadas de clases minoritarias...")
aumentar_train()

print(f"\n3. Creando data.yaml...")
crear_data_yaml()

print("✓ Augmentation completado")

In [ ]:
# Mostramos aleatoriamente algunos labels para comprobar que se han formado correctamente

lbl_dir = Path("datasets/rod_5cls_augmented/train/labels")
aug_labels = list(lbl_dir.glob("*_aug*.txt"))

if aug_labels:
    sample = aug_labels[0]
    
    print(f"Muestra: {sample.name}")
    with open(sample) as f:
        print(f.read())
else:
    print("No se encontraron labels aumentados")

In [ ]:
# Verificación del ROD-Dataset de 5 clases aumentado

DATASET = Path("datasets/rod_5cls_augmented")

print("Estructura:")
for split in ["train", "valid", "test"]:
    imgs = list((DATASET / split / "images").glob("*"))
    lbls = list((DATASET / split / "labels").glob("*.txt"))
    print(f"  {split:6s}: {len(imgs):>6d} imgs, {len(lbls):>6d} labels")

print("\nDistribución de clases (train):")
contador = Counter()
for lbl in (DATASET / "train" / "labels").glob("*.txt"):
    with open(lbl) as f:
        for line in f:
            if line.strip():
                contador[int(line.split()[0])] += 1

names = ['Obstaculo_Dinamico', 'Obstaculo_Fijo', 'Barrera_Arquitectonica', 'Bordillo', 'Elemento_Peatonal']
total = sum(contador.values())
for i, nombre in enumerate(names):
    c = contador.get(i, 0)
    print(f"  [{i}] {nombre:25s} {c:>7d}  ({c/total*100:5.2f}%)")
print(f"  {'':29s} {'TOTAL':>7s}: {total}")


mayor = max(contador.values())
menor = min(contador.values())
print(f"\nRatio mayor/menor: {mayor/menor:.1f}:1")

### Entrenamiento

In [18]:
# !rocm-smi  # Para GPUs AMD



============================================= ROCm System Management Interface =============================================
======================================================= Concise Info =======================================================
Device  Node  IDs              Temp    Power    Partitions          SCLK     MCLK     Fan     Perf    PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Edge)  (Avg)    (Mem, Compute, ID)                                                         
0       1     0x7590,   56445  38.0°C  27.0W    N/A, N/A, 0         1438Mhz  1258Mhz  15.69%  manual  200.0W  11%    7%    
1       2     0x164e,   24725  47.0°C  35.986W  N/A, N/A, 0         N/A      3000Mhz  0%      auto    N/A     45%    0%    
=================================================== End of ROCm SMI Log ====================================================


In [19]:
# !nvidia-smi  # Para GPUs de Nvidia

/bin/bash: línea 1: nvidia-smi: orden no encontrada


In [20]:
model = YOLO("yolo26s.pt")

In [ ]:
results = model.train(
    data="datasets/rod_14cls/data.yaml",
    epochs=40,
    imgsz=640,
    batch=-1,  # Lo adapta al 60% de la VRAM de tu GPU            
    patience=10,
    device=0,
    name="rod_14cls_yolo26s",
    close_mosaic=5,
    multi_scale=0.5,       # CLAVE: robustez a escalas
    degrees=10.0,          # rotaciones suaves
    perspective=0.0005,    # perspectiva suave
    cos_lr=True,           # Estabiliza la convergencia
)

### Métricas del entrenamiento

Durante el entrenamiento, Ultralytics muestra una serie de métricas que reflejan el rendimiento del modelo. Estas son las más importantes:

#### Pérdidas (Losses)

| Métrica | ¿Qué mide? | ¿Qué esperar? |
| :--- | :--- | :--- |
| **box_loss** | Error en la localización de las bounding boxes (coordenadas). | Debe bajar con las épocas. |
| **cls_loss** | Error en la clasificación de cada objeto (¿es un coche? ¿una persona?). | Debe bajar con las épocas. |
| **l1_loss** | Pérdida auxiliar de regresión que ayuda a estabilizar el entrenamiento. | Debe bajar con las épocas. |

#### Métricas de evaluación (Box)

| Métrica | ¿Qué mide? | ¿Qué esperar? |
| :--- | :--- | :--- |
| **Precision (P)** | De todos los objetos que el modelo detecta, ¿cuántos son correctos? | Cuanto más alto, mejor. Cercano a 1.0 es ideal. |
| **Recall (R)** | De todos los objetos reales, ¿cuántos detecta el modelo? | Cuanto más alto, mejor. Cercano a 1.0 es ideal. |
| **mAP50** | Precisión media con un umbral de solapamiento (IoU) del 50%. | Cuanto más alto, mejor. >0.90 es excelente. |
| **mAP50-95** | Precisión media promediada sobre umbrales de IoU de 50% a 95%. | Cuanto más alto, mejor. >0.70 es muy bueno. |

#### Otras métricas

| Métrica | ¿Qué mide? |
| :--- | :--- |
| **Instances** | Número de objetos anotados en el batch actual. |
| **GPU_mem** | Memoria de la GPU utilizada durante el entrenamiento. |
| **Size** | Tamaño de las imágenes de entrada (ej. 640x640). |

#### ¿Qué es el IoU?

El **IoU** (Intersection over Union) mide el solapamiento entre la bounding box predicha y la real. Va de 0 (sin solapamiento) a 1 (solapamiento perfecto). Un IoU de 0.5 significa que la predicción cubre al menos el 50% del objeto real.

#### ¿Qué es el mAP?

El **mAP** (mean Average Precision) es la métrica estándar para evaluar modelos de detección de objetos. Combina Precision y Recall en un solo número. Un mAP alto significa que el modelo detecta los objetos correctamente y con buena localización.

Si el mAP es bueno (>0.90) → Documentar y pasar a la demo.

Si el mAP es bajo (<0.80) → Analizar qué clases fallan y considerar:

    Más épocas.

    Data augmentation específico.

    Ajuste de hiperparámetros con model.tune().

Si la demo falla mucho → Documentar la limitación de dominio y proponer fine-tuning con imágenes locales como trabajo futuro.

In [ ]:
model = YOLO("runs/detect/rod_5cls_augmented_yolo26s/weights/best.pt")  # Cargamos el mejor modelo

In [ ]:
metrics = model.val(data="datasets/rod_5cls_augmented/data.yaml", split="test", name="val_rod_5cls_augmented")  # Test

In [ ]:
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("runs/detect/rod_5cls_augmented_yolo26s/confusion_matrix_normalized.png")  # Matriz de confusión normalizada
plt.figure(figsize=(14, 12))
plt.imshow(img)
plt.axis('off')
plt.show()

### Predicción con imágenes propias

In [ ]:
model = YOLO("runs/detect/rod_5cls_augmented_yolo26s/weights/best.pt")

for idx, name in model.names.items():
    print(f"  {idx}: {name}")
    
print(f"\nClases: {len(model.names)}")

In [ ]:
imgs_dir = Path("predicts/imgs/")
output_dir = Path("predicts/5cls-aug/")
output_dir.mkdir(exist_ok=True)

EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

for img_path in sorted(imgs_dir.iterdir()):
    if not img_path.is_file() or img_path.suffix.lower() not in EXTENSIONES:
        continue

    results = model(
        img_path, 
        conf=0.1, 
        device="cpu",
        imgsz=640  # Redimensión de las imágenes (2560 - ERROR GPU)
    )
    
    r = results[0]

    print(f"\n--- {img_path.name} ---")
    if len(r.boxes) == 0:
        print("  (sin detecciones)") 
    else:
        for box in r.boxes:
            clase = model.names[int(box.cls)]
            conf = float(box.conf)
            xyxy = box.xyxy[0].tolist()
            print(f"  {clase:20s} conf={conf:.2f}  bbox={[round(v) for v in xyxy]}")

    annotated = r.plot()
    cv2.imwrite(str(output_dir / img_path.name), annotated)